# SIA — Subject Identifiability Auditing (ICVGIP 2026)

Reproduces every table and figure of the paper on Kaggle (GPU: T4).

**Before running:** add the two datasets to the notebook (`ninadaithal/imagesoasis` and `mateuszbuda/lgg-mri-segmentation`), turn on **Internet** (ImageNet weights) and select a **GPU** accelerator.

In [ ]:
# Option A: the code is attached as a Kaggle dataset named 'sia-audit'
import os, shutil
if os.path.exists('/kaggle/input/sia-audit'):
    shutil.copytree('/kaggle/input/sia-audit', '/kaggle/working/sia-audit', dirs_exist_ok=True)
# Option B: clone from GitHub
# !git clone https://github.com/anuragtiwari575/SIA-subject-identifiability-audit.git /kaggle/working/sia-audit
%cd /kaggle/working/sia-audit
!pip install -q -e .

In [ ]:
OASIS = '/kaggle/input/imagesoasis/Data'
TCGA  = '/kaggle/input/lgg-mri-segmentation/kaggle_3m'
OUT   = '/kaggle/working/results'
import os; print(os.listdir(OASIS)); print(len(os.listdir(TCGA)), 'TCGA entries')

## Stage 1 — composition (Table 1, Fig. 1)

In [ ]:
!python -m sia.cli stage1 --oasis-root $OASIS --tcga-root $TCGA --out $OUT

## Stage 2 — paired protocol ablation + SIS of the standard encoder (Tables 2–3, Fig. 2)

≈ 30 min each on a T4 with the default 3 partitions.

In [ ]:
!python -m sia.cli stage2 --dataset oasis --oasis-root $OASIS --out $OUT

In [ ]:
!python -m sia.cli stage2 --dataset tcga --tcga-root $TCGA --out $OUT

Optional — the uncapped benchmark (ρ = 249, Table 2 † row). ≈ 1 h per run; one partition per protocol is enough.

In [ ]:
# !python -m sia.cli stage2 --dataset oasis --oasis-root $OASIS --k 0 --partitions 1 --out $OUT

## Redundancy sweep at fixed budget (Table 4, Fig. 3) — ≈ 1.5 h

In [ ]:
!python -m sia.cli budget --oasis-root $OASIS --out $OUT

## Stage 3 — cross-backbone / cross-benchmark (Table 8, Fig. 5) — ≈ 30 min

In [ ]:
!python -m sia.cli stage3 --oasis-root $OASIS --tcga-root $TCGA --out $OUT

## Interventions (Table 6, Fig. 4) — ≈ 3 h

Reduce `--interv-partitions` (e.g. `standard:5,grl:2,grl_convergence:1,mil:2,confusion:5`) for a shorter session; `stats` needs the same seeds for `confusion` and `standard`.

In [ ]:
!python -m sia.cli interventions --oasis-root $OASIS --out $OUT

## Statistics (Table 7), figures and report

In [ ]:
!python -m sia.cli stats --out $OUT
!python -m sia.cli figures --out $OUT
!python -m sia.cli report --out $OUT

In [ ]:
from IPython.display import Image, display, Markdown
display(Markdown(open(f'{OUT}/REPORT.md').read()))
for f in ['fig1_composition', 'fig3_confusion', 'fig2_budget', 'fig6_lambda', 'fig_dissociation']:
    display(Image(f'{OUT}/figures/{f}.png', width=700))

## Download everything (JSON results, figures, REPORT.md; checkpoints excluded)

In [ ]:
!cd /kaggle/working && zip -qr sia_results.zip results -x 'results/checkpoints/*' && ls -la sia_results.zip